In [1]:
import cdms
import numpy as np
import os, sys 
import ROOT
from cats.cdataframe import CDataFrame
from CDMSDataCatalog import CDMSDataCatalog
import supercuts
import glob
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
%matplotlib inline

CDMS = os.environ["CDMS"] # set in .bash_profile
stylesheet = os.path.join(CDMS,"scripts","stylesheets","blues.mplstyle")
plt.style.use(stylesheet)

sys.path.append(os.path.join(os.path.join(CDMS,"scripts")))
import setup

/usr/local/lib/python3.10/dist-packages/CDMSDataCatalog/CDMSDataCatalog.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Welcome to JupyROOT 6.28/10


In [3]:
from scipy.optimize import curve_fit
from uncertainties import *
from uncertainties.umath import exp
import math

def func(x, a, b):
    return a*x + b

def fit_logPulse(x, y, start, stop):
    popt, pcov = curve_fit(func, x[start:stop], y[start:stop])
    return popt[0], np.sqrt(np.diag(pcov))[0], popt[1], np.sqrt(np.diag(pcov))[1]

def str_with_err(value, error):
    digits = -int(math.floor(math.log10(error)))
    return "{0:.{2}f}({1:.0f})".format(value, error*10**digits, digits)

In [4]:
ufloat(2.2, 1.5) / ufloat(95.3, 1.5)

0.023084994753410287+/-0.015743962586580926

### Propagate error for Geant4 SourceSim result

Found that simulating 1E9 Cf-252 decays in the CUTE fridge results in 9208 Ge-70 + n $\to$ Ge-71 interactions.  
Conservative uncertainty for Geant4 hadronic physics is 4%.  
Poisson uncertainty for total count.

In [8]:
ufloat(3734, 189) / ufloat(4432, 67) * 37

31.17283393501805+/-1.6467133760806145

In [9]:
ufloat(0.037, 0.003) / ufloat(0.012, 0.016)

3.083333333333333+/-4.118705448062683

In [10]:
np.sqrt( (0.037 / 0.012**2 * 0.016)**2 + (1 / 0.012 * 0.003)**2)

4.118705448062683

In [11]:
(ufloat(3.6, 0.3) / ufloat(96.4, 0.3)) / (ufloat(1.1, 1.5) / ufloat(95.4, 1.4))

3.2387778196906827+/-4.425021154607636

In [12]:
ufloat(3.6, 0.3) / ufloat(96.4, 0.3)

0.03734439834024896+/-0.003114202466072628

In [13]:
ufloat(1.1, 1.5) / ufloat(95.4, 1.4)

0.011530398322851153+/-0.015724180903645

In [14]:
ufloat(4100, 640) * ufloat(0.911, 0.003)

3735.1+/-583.1697279523346

In [15]:
L_eff = ufloat(516, 25) / ufloat(445, 71)
K_eff = ufloat(4432, 67) / ufloat(3735, 583)

In [16]:
L_eff

1.1595505617977528+/-0.19334874273934657

In [17]:
K_eff

1.1866131191432396+/-0.1860863003987786

In [18]:
(L_eff + K_eff) / 2

1.1730818404704961+/-0.13417511646630914

In [19]:
def poisson_geant4_err(count):
    poisson_err = np.sqrt(count)
    geant4_err = count * 0.04

    err = np.sqrt(poisson_err ** 2 + geant4_err ** 2)
    decays = ufloat(count, err)
    
    return decays

In [20]:
print('Number of K shell decays', end=": ")
K_decays = poisson_geant4_err(int(8059))
print('{:+.3uS}'.format(K_decays))

print('Number of L shell decays', end=": ")
L_decays = poisson_geant4_err(int(959))
print('{:+.3uS}'.format(L_decays))

print('Number of M shell decays', end=": ")
M_decays = poisson_geant4_err(int(177))
print('{:+.3uS}'.format(M_decays))

print('Total number of decays', end=": ")
total_decays = K_decays + L_decays + M_decays
print('{:+.3uS}'.format(total_decays))

Number of K shell decays: +8059(335)
Number of L shell decays: +959.0(49.3)
Number of M shell decays: +177.0(15.1)
Total number of decays: +9195(339)


In [21]:
K_over_tot = K_decays / total_decays * 100
print('{:+.1uS}'.format(K_over_tot))

L_over_tot = L_decays / total_decays * 100
print('{:+.1uS}'.format(L_over_tot))

M_over_tot = M_decays / total_decays * 100
print('{:+.1uS}'.format(M_over_tot))

+87.6(7)
+10.4(6)
+1.9(2)


### Propagate errors in estimation of number of decays

[Cf-252 half-life](https://www.chemlin.org/isotope/californium-252)  
[Ge-71 half-life](https://www.chemlin.org/isotope/germanium-71)

In [22]:
# Z1 low bg 50V
series_list=['23240109_075338', '23240109_021236', 
'23240108_203134', '23231221_101235', 
'23231221_015705', '23231220_190923', 
'23231220_122140', '23231220_053358', 
'23231220_012745', '23231219_184002', 
'23231219_110331', '23231219_034952', 
'23231218_223530', '23231218_190035', 
'23231218_152721', '23231218_093255', 
'23231218_024511', '23231217_212512', 
'23231217_171613', 
'23231217_135018', '23231216_233807', 
'23231216_211119', '23231216_194929', 
'23231216_182937', '23231216_173436', 
'23231216_145300', '23231216_100125', 
'23231216_043946', '23231216_013604'] # Ge calibration  

In [23]:
ProdTag = 'CUTE_T3GeCalib_NxM_P4.0.0_V05-06_C0.3.6'

In [24]:
filepath = [f'/scratch/group/mitchcomp/CDMS/data/CDMS/CUTE/R37/Processed/Releases/{ProdTag}/Submerged/{ProdTag}_{i}.root' for i in series_list]

In [25]:
#filepath = np.sort(glob.glob(f'/scratch/group/mitchcomp/CDMS/data/CDMS/CUTE/R37/Processed/Releases/{ProdTag}/Submerged/{ProdTag}_????????_??????.root'))

In [26]:
det = 1 # detector number

df = CDataFrame("rqDir/zip"+str(det), filepath, friends = [[x+":rqDir/eventTree" for x in filepath]])

In [27]:
## Apply some basic data quality filters and get the RQs you're interested in
logfile = '"cute_tower3testing.log"'
df = df.Define("LEDLogFile", logfile) 
df = df.CDefine("LEDOn", supercuts.ledOn_old, ["EventTime", "LEDLogFile"])
df = df.Filter("!LEDOn")
df_filtered = df.Filters(["TriggerType == 1", "TriggerDetectorNum=="+str(det), "PTOFamps>0"])

In [28]:
RQs = (["SeriesNumber", "PTOFamps", "EventNumber", "EventTime"])
df_rqs = df_filtered.AsNumpy(RQs)

In [29]:
import datetime

In [30]:
y2s = 3.156e7
d2s = 86400
m2s = 60

# Cf252 radioactive properties
Cf252_halflife = ufloat(2.645 * y2s, 0.008 * y2s) # s
Cf252_lifetime = Cf252_halflife / np.log(2) # s
Cf252_lambda   = 1 / Cf252_lifetime # Hz

# Ge-71 radioactive properties
Ge71_halflife = ufloat(11.468 * d2s, 0.008 * d2s) # s
Ge71_lifetime = Ge71_halflife / np.log(2) # s
Ge71_lambda   = 1 / Ge71_lifetime

In [41]:
total_decays = ufloat(9195, np.sqrt(9195 + (0.04 * 9195)**2))

In [43]:
Cf252_activity = ufloat(37000, 37000 * 0.15)                          # activity of Cf252 at reference time; given in Hz
t_ref          = datetime.datetime(2020, 3, 15, 00, 0).timestamp()    # reference time of nominal activity; Unix time
activ_prob     = total_decays * 1e-9                                  # Ge activations per neutron emitted; found using sourcesim w/ CUTE geometry

# rate of Ge activation during exposure
A = Cf252_activity * activ_prob # activations / second

In [44]:
start_times = pd.read_csv('source_exposure_start_times.csv')
end_times = pd.read_csv('source_exposure_end_times.csv')

timestamp_err = 30 * m2s

exposures = {i: {} for i in range(8)}

for i in range(8):
    year_0, year_f   = start_times['year'][i], end_times['year'][i]
    month_0, month_f = start_times['month'][i], end_times['month'][i]
    day_0, day_f     = start_times['day'][i], end_times['day'][i]
    hour_0, hour_f   = start_times['hour'][i], end_times['hour'][i]
    min_0, min_f     = start_times['minute'][i], end_times['minute'][i]
    s_0, s_f         = start_times['second'][i], end_times['second'][i]
    exposures[i]['t0'] = ufloat(datetime.datetime(year_0, month_0, day_0, hour_0, min_0, s_0).timestamp(), timestamp_err)
    exposures[i]['tf'] = ufloat(datetime.datetime(year_f, month_f, day_f, hour_f, min_f, s_f).timestamp(), timestamp_err)
    exposures[i]['dt'] = exposures[i]['tf'] - exposures[i]['t0']

In [45]:
# find total decays with N = int[ A exp( -lambda_Cf252 (t - tref) ) ] dt
# t from t0 to T
def integrate_decays(T, t0):
    return (Cf252_activity * (-exp(Cf252_lambda * (t_ref - T)) + exp(Cf252_lambda * (t_ref - t0)))) / Cf252_lambda

In [46]:
#### starting with 0 decays ####
N_tot = 0

#### track change in number of Cf-252 decays during exposures ####
for i in range(len(exposures)):
    N = integrate_decays(exposures[i]['tf'], exposures[i]['t0'])

    N_tot += N

print('{:+.3uS} total Cf-252 decays'.format(N_tot))

+1.637(246)e+10 total Cf-252 decays


In [47]:
# find total activations with N = int[ A exp( -lambda_Cf252 (t + t0) ) * exp(-lambda_Ge71 (T - t) ) ] dt + N0 exp(-lambda_Ge71 T) 
# t from t0 to T
def integrate_activations(N, T, t0):
    return (A / (Cf252_lambda - Ge71_lambda) * ( exp(- Cf252_lambda * t0 - Ge71_lambda * T) - exp(-Cf252_lambda * (T + t0)) ) + 
            N * exp(-Ge71_lambda * T) )

In [49]:
#### starting with 0 nuclei activated ####
N = 0

#### track change in number of Ge-71 nuclei during and after exposures ####
for i in range(len(exposures)):
    N = integrate_activations(N, exposures[i]['dt'], exposures[i]['t0'] - t_ref)
    if (i != len(exposures) - 1):
        N = N * exp(-Ge71_lambda * (exposures[i+1]['t0'] - exposures[i]['tf']))
    else:
        # check change in number of Ge-71 during each series
        N_tot = 0
        for sn in np.unique(df_rqs['SeriesNumber']):
            snCut = df_rqs['SeriesNumber'] == sn
            evtTimes = df_rqs['EventTime'][snCut]
            sn_start = min(evtTimes)
            sn_end = max(evtTimes)

            N_start = N * exp(-Ge71_lambda * (sn_start - (exposures[len(exposures) - 1]['t0'] + exposures[len(exposures) - 1]['dt'])))
            N_end = N * exp(-Ge71_lambda * (sn_end - (exposures[len(exposures) - 1]['t0'] + exposures[len(exposures) - 1]['dt'])))
            #print(f'{np.round(N_start - N_end)} Ge-71 events in series {int(sn)}')

            N_tot += N_start - N_end

print('{:+.3uS} total Ge-71 events'.format(N_tot))
print('{:+.3uS} K shell events'.format(N_tot * K_over_tot / 100))
print('{:+.3uS} L shell events'.format(N_tot * L_over_tot / 100))
print('{:+.3uS} M shell events'.format(N_tot * M_over_tot / 100))

+4680(733) total Ge-71 events
+4102(643) K shell events
+488.2(81.7) L shell events
+90.1(16.3) M shell events


In [37]:
ufloat(177, 177 * 0.04) + ufloat(959, 959 * 0.04) + ufloat(8059, 8059 * 0.04)

9195.0+/-324.71154214163687

In [50]:
N_tot * L_over_tot / 100 * ufloat(9150, 95.76) / 10000

446.6613313142392+/-74.87221760062621

In [51]:
N_tot * K_over_tot / 100 * ufloat(91058, 301.89) / 100000

3735.4069097180904+/-586.0005498266572

In [61]:
ufloat(516, 24) / ufloat(447, 34) * 37

42.711409395973156+/-3.807993835007216

In [62]:
ufloat(4432, 67) / ufloat(3735, 188) * 37

43.90468540829987+/-2.307446164218312

In [57]:
np.sqrt(75**2 - (0.15 * 447)**2)

33.60502194613181

In [58]:
np.sqrt(591**2 - (0.15 * 3735)**2)

188.15136858391438